In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA
import pickle

from analysis_village.cc1pi.var_configs import *

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

from analysis_village.cc1pi.CutMasks.MaskUtils import *
# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)


from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.makedf import make_cc1pidf
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils
from analysis_village.cc1pi.BDTs import BDTTrainingUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']

#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_lowE_CV.df", keys2load, 100)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']

#OLD OLD OLD OLD
#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']


In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))

intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics data gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))

In [ ]:
mc_evt_df = concat_shift_first_index(mc_bnb_evt_df,mc_in_time_cosmics_evt_df)
mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
evt_df = concat_shift_first_index(mc_evt_df,mc_bnb_lowE_evt_df)

In [ ]:
def is_MIP_candidate_mask(df):
    is_primary_mask = (df.pfp.parent_is_primary == True) & (df.pfp.dist_to_vertex < CTE.max_primary_distance_to_vertex)
    is_track_mask = (df.pfp.trk.len > CTE.min_track_lenght) & (df.pfp.trackScore > CTE.min_track_score)
    track_mask = is_primary_mask & is_track_mask

    len_mask = df.pfp.trk.len > CTE.MIP_candidate_min_TL
    chi2_mask = (df.pfp.trk.chi2pid.best.chi2_muon < CTE.MIP_candidate_max_muon_score) & (df.pfp.trk.chi2pid.best.chi2_proton > CTE.MIP_candidate_min_proton_score)
    return track_mask & chi2_mask & len_mask
    
def is_primary_track_mask(df):
    is_primary_mask = (df.pfp.parent_is_primary == True) & (df.pfp.dist_to_vertex < CTE.max_primary_distance_to_vertex)
    is_track_mask = (df.pfp.trk.len > CTE.min_track_lenght) & (df.pfp.trackScore > CTE.min_track_score)
    return is_primary_mask & is_track_mask

In [ ]:
def proton_BDT_cut_mask(df, group_levels):
    BDT_proton_df = df[(is_MIP_candidate_mask(df)) & (df.pfp.trk.bdt_proton_score > 1.1)]
    
    # Count how many pfps per slice
    candidate_counts = BDT_proton_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = candidate_counts[candidate_counts == 2].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask

In [ ]:
def proton_condition_mask(df, group_levels):
    # 1. Create boolean flags for the two conditions
    # Flag 1: Is it a primary track we care about?
    chi2_mask = (df.pfp.trk.chi2pid.best.chi2_muon < CTE.MIP_candidate_max_muon_score) & \
                (df.pfp.trk.chi2pid.best.chi2_proton > CTE.MIP_candidate_min_proton_score)
    is_relaxed_mip = is_primary_track_mask(df) & (~chi2_mask)
    
    # Flag 2: Is it a proton?
    is_proton = df.pfp.trk.chi2pid.best.chi2_proton < df.pfp.trk.chi2pid.best.chi2_muon
    
    # 2. Group by slice levels and sum the flags
    # Summing a boolean Series treats True as 1 and False as 0.
    counts_df = pd.DataFrame({
        'total_mip': is_relaxed_mip.astype(int),
        'proton_mip': (is_relaxed_mip & is_proton).astype(int)
    }).groupby(level=group_levels).sum()

    # 3. Determine validity
    # A slice is valid if total == proton (this includes 0 == 0)
    valid_slices_mask = (counts_df['total_mip'] == counts_df['proton_mip'])
    valid_slice_indices = counts_df.index[valid_slices_mask]

    # 4. Map back to the original full dataframe
    # We drop the PFP level to align the full index with our slice-level decisions
    final_mask = df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slice_indices)
    
    return final_mask

In [ ]:
def get_muon_pion_slice_mask(df):
    """
    Returns a mask for slices that contain exactly one true muon MIP candidate 
    and exactly one true pion MIP candidate.
    """
    pdg_col = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
    SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

    # 1. Get the particle-level MIP candidate mask
    # This uses your existing function logic
    mip_particles = is_MIP_candidate_mask(df)

    # 2. Identify True Muons and True Pions that are ALSO MIP candidates
    is_muon_mip = (df[pdg_col].abs() == 13) & mip_particles
    is_pion_mip = (df[pdg_col].abs() == 211) & mip_particles

    # 3. Count these specific candidates per slice
    # transform('sum') broadcasts the count to all rows in the slice
    muon_mip_count = is_muon_mip.groupby(level=SLICE_LEVELS).transform('sum')
    pion_mip_count = is_pion_mip.groupby(level=SLICE_LEVELS).transform('sum')
    
    # 4. Final condition: The slice must have exactly one of each MIP-quality particle
    slice_condition = (muon_mip_count == 1) & (pion_mip_count == 1)

    return slice_condition

In [ ]:
evt_df[('slc', 'cut', 'is_perfect_cc1pi', '', '', '')] = get_muon_pion_slice_mask(evt_df)
evt_df[('slc', 'cut', 'proton_condition', '', '', '')] = proton_condition_mask(evt_df, ['__ntuple', 'entry', 'rec.slc..index'])
slc_df = evt_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()

In [ ]:
'''
obvious_cosmic_mask_evt = slc_df.slc.cut.obvious_cosmic
t0_mask_evt = slc_df.slc.cut.t0
is_inside_FV_mask_evt = slc_df.slc.cut.inside_FV
nu_score_mask_evt = slc_df.slc.cut.nu_score
track_mask_evt = slc_df.slc.cut.track
shower_mask_evt = slc_df.slc.cut.shower 
chi2_mask_evt = slc_df.slc.cut.MIP_candidates 
angle_mask_evt = slc_df.slc.cut.angle 
proton_BDT_mask_evt = slc_df.slc.cut.proton_BDT
containment_mask_evt = slc_df.slc.cut.containment 
michel_mask_evt = slc_df.slc.cut.michel 
extra_pion_mask_evt =slc_df.slc.cut.extra_pion 
proton_condidtion_mask_evt =slc_df.slc.cut.proton_condition 
energy_mask_evt = slc_df.slc.cut.energy

cut_sequence = [
    ("cosmic", obvious_cosmic_mask_evt),
    ("t0", t0_mask_evt),
    ("FV", is_inside_FV_mask_evt),
    ("nu_score", nu_score_mask_evt),
    ("track", track_mask_evt),
    ("shower", shower_mask_evt),
    ("chi2", chi2_mask_evt),
    ("angle", angle_mask_evt),
    ("proton_BDT", proton_BDT_mask_evt),
    ("containment", containment_mask_evt),
    ("michel", michel_mask_evt),
    ("extra_pion", extra_pion_mask_evt),
    ("energy", energy_mask_evt),
  #  ("proton_condidtion_mask_evt", proton_condidtion_mask_evt)
]

current_mask_evt = None
mc_cumulative_masks = {}
for name, mask in cut_sequence:
    if current_mask_evt is None:
        current_mask_evt = mask
    else:
        current_mask_evt = current_mask_evt & mask
    mc_cumulative_masks[name] = current_mask_evt
'''
mc_cumulative_masks = build_event_cumulative_masks(slc_df, plot_sideband = False)

In [ ]:
## 1. Define the Signal Condition and get the Initial Weighted Total
# signal_condition_slc is a boolean mask (True/False)
signal_condition_slc = (slc_df.truth.nu_categ == "CC1pi")

# Weighted total signal = (Boolean Mask * Weights).sum()
n_total_signal = (signal_condition_slc * slc_df[pot_weight_col]).sum()

print(f"{'Cut Name':<20} | {'Weighted Sig':<12} | {'Eff':<9} | {'Pur':<9} | {'Pur*Eff':<9}")
print("-" * 75)

for name, mask in mc_cumulative_masks.items():
    # 2. Map the mask to the slice level
    # We use .first() to get the mask value for the slice
    mask_slc = (
        mask.groupby(level=['__ntuple', 'entry', 'rec.slc..index'])
        .first()
        .reindex(slc_df.index, fill_value=False)
    )
    
    # 3. Calculate Weighted Counts
    # Signal passing: (Mask is True AND Category is Signal) * Weights
    n_signal_passing = ((mask_slc & signal_condition_slc) * slc_df[pot_weight_col]).sum()
    
    # Total passing: (Mask is True) * Weights
    n_total_passing = (mask_slc * slc_df[pot_weight_col]).sum()
    
    # 4. Calculate Stats
    eff = n_signal_passing / n_total_signal if n_total_signal > 0 else 0
    pur = n_signal_passing / n_total_passing if n_total_passing > 0 else 0
    pur_eff = pur * eff
    
    # 5. Print results (using :.2f for weighted counts since they aren't integers)
    print(f"{name:<20} | {n_signal_passing:<12.2f} | {eff:.2%}  | {pur:.2%}  | {pur_eff:.4f}")

In [ ]:

config_n_protons = HistogramConfig(
    data_column=('slc','measure_var','num_protons','','',''),
    bins=np.linspace(0, 2, 4),
    xlabel='num protons',
    ylabel='Entries',
    title='num protons'
)

plot_stacked_histogram(
    slc_df[mc_cumulative_masks["energy"]],
    config=config_n_protons,
    type_column=('truth','nu_categ','','','',''),
    first_per_slice = False
)


In [ ]:

def plot_confusion_matrix(
    cm,
    x_labels,   # True labels
    y_labels,   # Reco labels
    cmap=sunset_cmap,
):
    cm = np.asarray(cm, dtype=float)

    # -------------------------
    # Normalizations
    # -------------------------
    # Purity (row-normalized): Sum over True for a given Reco
    row_denom = cm.sum(axis=1, keepdims=True)
    cm_purity = np.divide(cm, row_denom, where=row_denom > 0) * 100

    # Efficiency (column-normalized): Sum over Reco for a given True
    col_denom = cm.sum(axis=0, keepdims=True)
    cm_eff = np.divide(cm, col_denom, where=col_denom > 0) * 100

    # Eff * Pur
    cm_effpur = cm_purity * cm_eff / 100.0

    # -------------------------
    # Plot
    # -------------------------
    fig, axes = plt.subplots(
        1, 3, figsize=(18, 5), sharey=True, constrained_layout=True
    )

    matrices = [
        (cm_purity, "Purity", True),
        (cm_eff, "Efficiency", True),
        (cm_effpur, "Eff × Pur", False),
    ]

    for ax, (cm_plot, title, show_counts) in zip(axes, matrices):
        im = ax.imshow(cm_plot, cmap=cmap, vmin=0, vmax=100, origin="lower", aspect='auto')

        # FIX: Use specific lengths for each axis based on the actual labels provided
        ax.set_xticks(np.arange(len(x_labels)))
        ax.set_yticks(np.arange(len(y_labels)))
        
        ax.set_xticklabels(x_labels, rotation=45)
        ax.set_yticklabels(y_labels)

        ax.set_xlabel("True")
        ax.set_title(title)

        # Loop through the matrix shape, not the label list length
        for i in range(cm.shape[0]):    # Row index (Reco)
            for j in range(cm.shape[1]): # Column index (True)
                value = cm_plot[i, j]

                if show_counts:
                    txt = f"{int(cm[i,j])}\n{value:.1f}%"
                else:
                    txt = f"{value:.1f}%"

                text_color = "white" if value > 60 else "black"
                ax.text(
                    j, i, txt,
                    ha="center", va="center",
                    color=text_color,
                    fontsize=10
                )

    axes[0].set_ylabel("Reco")
    cbar = fig.colorbar(im, ax=axes.ravel().tolist())
    cbar.set_label("Percentage (%)")

    plt.show()

In [ ]:
from analysis_village.cc1pi.HelperFunctions import HelperFunctions

In [ ]:

def IsNu(df):
    is_numu = abs(df.pdg) == 14
    is_nue = abs(df.pdg) == 12
    return is_numu | is_nue   
    
def TruthInFV(data):
    xmin = -200 + CTE.min_distance_to_wall_x_y
    xmax = 200 - CTE.min_distance_to_wall_x_y
    ymin = -200 + CTE.min_distance_to_wall_x_y
    ymax = 200 - CTE.min_distance_to_wall_x_y
    zmin = CTE.min_distance_to_first_z_wall
    zmax = 500 - CTE.min_distance_to_last_z_wall
    
    pass_fv = (data.x > xmin) & (data.x < xmax) & (data.y > ymin) & (data.y < ymax) & (data.z < zmax) & (data.z > zmin)
    return pass_fv

def isCC1Pi(df): # definition
    is_1pi1mu = (df.nmu_P_100MeV_3000MeV == 1) & (df.npi_P_130MeV_800MeV == 1) & (df.npi_P_85MeV_10000MeV == 1)
    is_NpiNmuNnNp = df.nprim - df.nmu - df.npi - df.np - df.nn == 0

    # Initialize full theta mask (False by default)
    is_theta = pd.Series(False, index=df.index)

    # Only compute angles where needed
    df_sel = df.loc[is_1pi1mu]
    
    if len(df_sel) > 0:
        cpi_vec = df_sel.loc[:, ('cpi','genp',['x','y','z'])].to_numpy()
        mu_vec  = df_sel.loc[:, ('mu','genp',['x','y','z'])].to_numpy()
     

        mu_mag  = np.linalg.norm(mu_vec, axis=1)
        cpi_mag = np.linalg.norm(cpi_vec, axis=1)
        dot     = np.sum(mu_vec * cpi_vec, axis=1)

        cos_theta = dot / np.clip(mu_mag * cpi_mag, 1e-12, None)
        theta     = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        
        # Assign back using the SAME index subset
        is_theta.loc[df_sel.index] = theta < CTE.max_angle_between_candidates
    
    #return is_1pi1mu & is_NpiNmuNnNp & is_theta & is_mu_contained
    return is_1pi1mu & is_NpiNmuNnNp & is_theta

In [ ]:
# 1. Define the keys
proto_key = ('truth', 'nu_categ_proton_reduced', '', '', '', '')
wgt_key = ('slc', 'wgt', '', '', '', '')

# 2. Get unique values (using the energy mask as you had it)
mask = mc_cumulative_masks["energy"]# & (slc_df.truth.nu_categ != "CC1pi")
unique_values = slc_df.loc[mask, proto_key].unique()
#unique_values = slc_df.loc[mc_cumulative_masks["energy"], proto_key].unique()

weighted_distribution = (
    slc_df[mask].groupby(proto_key)[[wgt_key]]
    .sum()
    .sort_values(by=wgt_key, ascending=False)
)

print(f"--- Weighted Distribution for Cut: 'energy' ---")
print(f"{'Category':<25} | {'Weighted Events':>15}")
print("-" * 45)

for val, row in weighted_distribution.iterrows():
    # val is the category name, row[wgt_key] is the sum of weights
    category_name = str(val) 
    print(f"{category_name:<25} | {row[wgt_key]:>15.2f}")

# 3. Calculate the total for a quick sanity check
total_weighted = weighted_distribution[wgt_key].sum()
print("-" * 45)
print(f"{'TOTAL':<25} | {total_weighted:>15.2f}")

In [ ]:
HelperFunctions.print_purity(slc_df[mc_cumulative_masks["energy"]], ('truth','nu_categ','','','',''))

In [ ]:
import numpy as np
import pandas as pd
from analysis_village.cc1pi.systematics.utils import *

# 1. Define Column Keys
true_categ_key   = ('truth', 'nu_categ', '', '', '', '')
true_protons_key = ('truth', 'nu_categ_proton_reduced', '', '', '', '')
reco_protons_key = ('slc', 'measure_var', 'num_protons', '', '', '')
pot_weight_col   = ('slc', 'wgt', '', '', '', '')

# --- Logic for Truth & Reco (Temporary Columns) ---
is_cc1pi = slc_df[true_categ_key] == "CC1pi"
nu_protons = slc_df[true_protons_key]
passes_cuts = mc_cumulative_masks["energy"]
n_p_reco = slc_df[reco_protons_key]

choices = ["bkg", "0p_CC1Pi", "1p_CC1Pi", "plus2p_CC1Pi"]

# We assign these as simple strings (not in the MultiIndex)

slc_df['tmp_true'] = np.select([
    (~is_cc1pi),
    (is_cc1pi & (nu_protons == "0p_CC1Pi")),
    (is_cc1pi & (nu_protons == "1p_CC1Pi")),
    (is_cc1pi & (nu_protons == "plus2p_CC1Pi"))
], choices, default="bkg")

#slc_df['tmp_true'] = slc_df[true_protons_key].astype(str)

slc_df['tmp_reco'] = np.select([
    (~passes_cuts),
    (passes_cuts & (n_p_reco == 0)),
    (passes_cuts & (n_p_reco == 1)),
    (passes_cuts & (n_p_reco > 1))
], choices, default="bkg")

# 2. AGGREGATE WEIGHTS
# We sum the weights grouped by our new categories
weighted_counts = slc_df.groupby(['tmp_true', 'tmp_reco'])[[pot_weight_col]].sum()
#print(weighted_counts)

# 3. BUILD MATRIX MANUALLY FROM THE GROUPBY OBJECT
labels = ["bkg", "0p_CC1Pi", "1p_CC1Pi", "plus2p_CC1Pi"]
pretty_labels = ["Background", r"$CC1\pi 0p$", r"$CC1\pi 1p$", r"$CC1\pi 2p+$"]
label_to_idx = {l: i for i, l in enumerate(labels)}

weighted_cm = np.zeros((len(labels), len(labels)))

# Instead of iterrows (which struggles with MultiIndex columns), 
# we iterate through the GroupBy index directly.
for (t_cat, r_cat), row in weighted_counts.iterrows():
    # Ensure we are dealing with strings, not Series
    t_str = str(t_cat)
    r_str = str(r_cat)
    
    if t_str in label_to_idx and r_str in label_to_idx:
        t_idx = label_to_idx[t_str]
        r_idx = label_to_idx[r_str]
        # Use .iloc[0] to get the scalar value from the weight Series
        weighted_cm[t_idx, r_idx] = row.iloc[0]

# 4. PLOT
cm_to_plot = weighted_cm.T 
fig = ConfusionMatricesUtils.plot_confusion_matrix(cm_to_plot, pretty_labels, y_labels=pretty_labels)


file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/ProtonSeparation"
os.makedirs(file_dir, exist_ok=True)  # create directory if needed

fig.savefig(file_dir + "/confusion_matrix_proton_selection", bbox_inches='tight', dpi=dpi)
plt.show()


# Pion Separation Stuff

In [ ]:

print(dpi)
# 6. Save and Show
file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/MuonPionSeparation"
os.makedirs(file_dir, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# 1. Configuration and Labels
true_labels = ["other", "muon", "pion", "proton", "shower"]
pretty_true_labels = ["other", r"$\mu$", r"$\pi$", "p", r"$e/\gamma$"]
reco_labels = ["muon", "pion"]
pretty_reco_labels = [r"$\mu$", r"$\pi$"]
pot_weight_col = ('slc', 'wgt', '', '', '', '') 

# Map labels to matrix indices
true_label_to_idx = {l: i for i, l in enumerate(true_labels)}
reco_label_to_idx = {l: i for i, l in enumerate(reco_labels)}

# Mapping for True IDs
def get_clean_ids(series):
    s = series.replace(['inelastic pion', 'stopping pion'], 'pion')
    return s.where(s.isin(true_labels), "other")

# Define the different containment scenarios to process
# Format: (Title Suffix, Filter Mask, Filename)
scenarios = [
    ("All", True, "all"),
    ("Contained", (slc_df.slc.measure_var.muon_contained == True), "contained"),
    ("Exiting", (slc_df.slc.measure_var.muon_contained == False), "exiting")
]

# 2. Main Loop through Scenarios
for title_suffix, containment_mask, file_suffix in scenarios:
    print(f"\n--- Processing Scenario: {title_suffix} ---")
    
    # Apply global energy mask + specific containment mask
    full_mask = mc_cumulative_masks["energy"] & containment_mask
    
    # Reduce to one entry per slice
    slice_df = slc_df[full_mask].groupby(
        level=['__ntuple', 'entry', 'rec.slc..index']
    ).first().copy()

    if slice_df.empty:
        print(f"Warning: No events found for scenario {title_suffix}. Skipping.")
        continue

    # Initialize weighted matrix for this scenario
    weighted_cm = np.zeros((len(true_labels), len(reco_labels)))

    reco_tasks = [
        ('muon', ('slc','measure_var','mu_true_p_type','','','')),
        ('pion', ('slc','measure_var','pi_true_p_type','','',''))
    ]

    # 3. Accumulate Weights
    for r_label, true_p_type_key in reco_tasks:
        slice_df['tmp_true'] = get_clean_ids(slice_df[true_p_type_key])
        weighted_counts = slice_df.groupby('tmp_true')[[pot_weight_col]].sum()
        
        r_idx = reco_label_to_idx[r_label]
        
        for t_cat, row in weighted_counts.iterrows():
            t_str = str(t_cat)
            if t_str in true_label_to_idx:
                t_idx = true_label_to_idx[t_str]
                weighted_cm[t_idx, r_idx] += row.iloc[0]

    # 4. Plotting
    cm_to_plot = weighted_cm.T 
    fig = ConfusionMatricesUtils.plot_confusion_matrix(
        cm_to_plot, 
        pretty_true_labels, 
        y_labels=pretty_reco_labels
    )
    
    # Add scenario to title for clarity
    fig.suptitle(f"Muon/Pion Separation ({title_suffix})", fontsize=16)

    # 5. Save and Show
    save_path = os.path.join(file_dir, f"confusion_matrix_muon_pion_separation_{file_suffix}.pdf")
    fig.savefig(save_path, bbox_inches='tight', dpi=dpi)
    print(f"Saved: {save_path}")
    plt.show()

# Perfect Sample

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# 1. Configuration and Labels
true_labels = ["muon", "pion"]
pretty_true_labels = [r"$\mu$", r"$\pi$"]
reco_labels = ["muon", "pion"]
pretty_reco_labels = [r"$\mu$", r"$\pi$"]
pot_weight_col = ('slc', 'wgt', '', '', '', '') 

# Map labels to matrix indices
true_label_to_idx = {l: i for i, l in enumerate(true_labels)}
reco_label_to_idx = {l: i for i, l in enumerate(reco_labels)}

# Mapping for True IDs
def get_clean_ids(series):
    s = series.replace(['inelastic pion', 'stopping pion'], 'pion')
    return s.where(s.isin(true_labels), "other")

# Define the different containment scenarios to process
# Format: (Title Suffix, Filter Mask, Filename)
scenarios = [
    ("All", True, "all"),
    ("Contained", (slc_df.slc.measure_var.muon_contained == True), "contained"),
    ("Exiting", (slc_df.slc.measure_var.muon_contained == False), "exiting")
]

# 2. Main Loop through Scenarios
for title_suffix, containment_mask, file_suffix in scenarios:
    print(f"\n--- Processing Scenario: {title_suffix} ---")
    
    # Apply global energy mask + specific containment mask
    full_mask = slc_df.slc.cut.is_perfect_cc1pi & (slc_df.truth.nu_categ == "CC1pi") & mc_cumulative_masks["energy"]  &  containment_mask
    
    # Reduce to one entry per slice
    slice_df = slc_df[full_mask].groupby(
        level=['__ntuple', 'entry', 'rec.slc..index']
    ).first().copy()

    if slice_df.empty:
        print(f"Warning: No events found for scenario {title_suffix}. Skipping.")
        continue

    # Initialize weighted matrix for this scenario
    weighted_cm = np.zeros((len(true_labels), len(reco_labels)))

    reco_tasks = [
        ('muon', ('slc','measure_var','mu_true_p_type','','','')),
        ('pion', ('slc','measure_var','pi_true_p_type','','',''))
    ]

    # 3. Accumulate Weights
    for r_label, true_p_type_key in reco_tasks:
        slice_df['tmp_true'] = get_clean_ids(slice_df[true_p_type_key])
        weighted_counts = slice_df.groupby('tmp_true')[[pot_weight_col]].sum()
        
        r_idx = reco_label_to_idx[r_label]
        
        for t_cat, row in weighted_counts.iterrows():
            t_str = str(t_cat)
            if t_str in true_label_to_idx:
                t_idx = true_label_to_idx[t_str]
                weighted_cm[t_idx, r_idx] += row.iloc[0]

    # 4. Plotting
    cm_to_plot = weighted_cm.T 
    fig = ConfusionMatricesUtils.plot_confusion_matrix(
        cm_to_plot, 
        pretty_true_labels, 
        y_labels=pretty_reco_labels
    )
    
    # Add scenario to title for clarity
    fig.suptitle(f"Muon/Pion Separation ({title_suffix})", fontsize=16)

    # 5. Save and Show
    save_path = os.path.join(file_dir, f"confusion_matrix_muon_pion_separation_only_mu_pi_{file_suffix}.pdf")
    fig.savefig(save_path, bbox_inches='tight', dpi=dpi)
    print(f"Saved: {save_path}")
    plt.show()